# Train TextCNN (Kaggle T4)

Notebook huan luyen TextCNN de phan loai `macro_domain` tu cau hoi phap luat.

This notebook trains a Kim-style TextCNN for Vietnamese legal **macro_domain** classification from the question text; Kim (2014)-style choices (optimizer, loss, max-norm, sampling) are noted in inline comments in the code cells below.

In [1]:
import os
import re
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

Device: cuda
GPU: Tesla T4


In [2]:
def detect_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/hngphtrn/legals/data/data_ready'),
        Path('/kaggle/input/datasets/hngphtrn/legals/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data_ready'),
        Path('/kaggle/working/vnlegal-rag/data/data_ready'),
        Path('/kaggle/working/legals/data/data_ready'),
        Path('/kaggle/working/data/data_ready'),
        Path('data/data_ready'),
        Path('../data/data_ready'),
    ]
    for p in candidates:
        if p.exists() and (p / 'qa_train.csv').exists():
            return p
    raise FileNotFoundError('Khong tim thay data_ready. Hay kiem tra duong dan dataset tren Kaggle.')

DATA_DIR = detect_data_dir()
ARTIFACT_DIR = Path('/kaggle/working/textcnn_artifacts' if Path('/kaggle').exists() else 'artifacts/textcnn')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

DATA_DIR = /kaggle/input/datasets/hngphtrn/legals/data_ready
ARTIFACT_DIR = /kaggle/working/textcnn_artifacts


In [3]:
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='\t')
qa_val = pd.read_csv(DATA_DIR / 'qa_val.csv', sep='\t')
qa_test = pd.read_csv(DATA_DIR / 'qa_test.csv', sep='\t')

required_cols = {'question', 'macro_domain'}
for name, df in [('train', qa_train), ('val', qa_val), ('test', qa_test)]:
    miss = required_cols - set(df.columns)
    if miss:
        raise ValueError(f'{name} missing columns: {miss}')

print('train:', qa_train.shape, 'val:', qa_val.shape, 'test:', qa_test.shape)
qa_train[['question','macro_domain']].head()

train: (23311, 14) val: (2841, 14) test: (2991, 14)


,question,macro_domain
0,"Theo Điều 5 Luật Địa chất và Khoáng sản, hội n...","Industry, Resources & Environment"
1,"Theo Điều 5 của Luật, khi thực hiện hội nhập v...","Industry, Resources & Environment"
2,Giả sử có một tranh chấp quốc tế phát sinh liê...,"Industry, Resources & Environment"
3,"Theo Điều 2 của Luật Địa chất và Khoáng sản, ""...","Industry, Resources & Environment"
4,Hãy phân tích sự khác biệt và mối quan hệ giữa...,"Industry, Resources & Environment"


In [4]:
# De co tokenizer tieng Viet tot hon, nen cai pyvi (on dinh hon tren Kaggle/Python 3.12).
!pip -q install pyvi
# Neu muon dung underthesea o moi truong khac:
# !pip -q install underthesea==6.8.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.1 MB/s eta 0:00:00


In [5]:
TOKENIZER_BACKEND = 'whitespace'
PYVI_IMPORT_ERROR = None
UNDERTHESEA_IMPORT_ERROR = None

# Uu tien pyvi (on dinh tren Kaggle/Python 3.12), neu khong co thi moi thu underthesea.
# prefer pyvi for Vietnamese word segmentation; fall back to underthesea, then whitespace split.
try:
    from pyvi import ViTokenizer
    TOKENIZER_BACKEND = 'pyvi'
except Exception as exc_pyvi:
    PYVI_IMPORT_ERROR = exc_pyvi
    try:
        from underthesea import word_tokenize
        TOKENIZER_BACKEND = 'underthesea'
    except Exception as exc_uts:
        UNDERTHESEA_IMPORT_ERROR = exc_uts
        TOKENIZER_BACKEND = 'whitespace'

print('Tokenizer backend:', TOKENIZER_BACKEND)
if TOKENIZER_BACKEND != 'pyvi' and PYVI_IMPORT_ERROR is not None:
    print('pyvi import error:', repr(PYVI_IMPORT_ERROR))
if TOKENIZER_BACKEND == 'whitespace' and UNDERTHESEA_IMPORT_ERROR is not None:
    print('underthesea import error:', repr(UNDERTHESEA_IMPORT_ERROR))


def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def tokenize(text: str):
    text = normalize_text(text)
    if TOKENIZER_BACKEND == 'pyvi':
        return [t for t in ViTokenizer.tokenize(text).split() if t]
    if TOKENIZER_BACKEND == 'underthesea':
        return [t for t in word_tokenize(text, format='list') if t]
    return text.split()

Tokenizer backend: pyvi


In [6]:
PAD = '<PAD>'
UNK = '<UNK>'
# cap vocabulary at ~10k most frequent train tokens (+ PAD/UNK); increase if train unique types exceed cap.
MAX_VOCAB = 10000
# fixed sequence length after tokenization (truncate right, pad with PAD id).
MAX_LEN = 128

print('TOKENIZER_BACKEND=', TOKENIZER_BACKEND)
print('MAX_LEN=', MAX_LEN)

counter = Counter()
for q in qa_train['question'].astype(str).tolist():
    counter.update(tokenize(q))

unique_tokens_train = len(counter)
total_tokens_train = sum(counter.values())

token_lens = [len(tokenize(q)) for q in qa_train['question'].astype(str).tolist()]
max_len_tokens = max(token_lens)
p95_len = float(np.percentile(token_lens, 95))
p99_len = float(np.percentile(token_lens, 99))
pct_truncated = 100.0 * sum(1 for L in token_lens if L > MAX_LEN) / len(token_lens)

k_cap = min(MAX_VOCAB - 2, unique_tokens_train)
coverage_topk = (
    sum(c for _, c in counter.most_common(k_cap)) / total_tokens_train
    if total_tokens_train
    else 0.0
)

print('unique_tokens_train:', unique_tokens_train)
print(
    'max_len_tokens:', max_len_tokens,
    '| p95:', round(p95_len, 2),
    '| p99:', round(p99_len, 2),
)
print(f'pct samples truncated (len > MAX_LEN={MAX_LEN}): {pct_truncated:.2f}%')
print('coverage_topk (token mass in vocab top-K, K=%d): %.4f' % (k_cap, coverage_topk))

vocab_tokens = [PAD, UNK] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(vocab_tokens)}
itos = {i: w for w, i in stoi.items()}

labels = sorted(qa_train['macro_domain'].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

print('Vocab size:', len(stoi), '| Num labels:', len(labels))

TOKENIZER_BACKEND= pyvi
MAX_LEN= 128
unique_tokens_train: 5580
max_len_tokens: 163 | p95: 71.0 | p99: 92.0
pct samples truncated (len > MAX_LEN=128): 0.07%
coverage_topk (token mass in vocab top-K, K=5580): 1.0000
Vocab size: 5582 | Num labels: 8


In [7]:
# map OOV tokens to UNK id; pad to max_len; literal keys match PAD/UNK strings above.
def encode_text(text, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [stoi.get(t, stoi["<UNK>"]) for t in tokens][:max_len]
    if len(ids) < max_len:
        ids += [stoi["<PAD>"]] * (max_len - len(ids))
    return ids

class QADomainDataset(Dataset):
    def __init__(self, df):
        self.questions = df['question'].astype(str).tolist()
        self.labels = [label2id[x] for x in df['macro_domain'].tolist()]

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode_text(self.questions[idx]), dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

In [8]:
# batch size for this project (not specified in Kim 2014).
BATCH_SIZE = 50
import sys
_num_workers = 0 if sys.platform == 'win32' else 2

train_ds = QADomainDataset(qa_train)
val_ds = QADomainDataset(qa_val)
test_ds = QADomainDataset(qa_test)

train_label_ids = np.array([label2id[x] for x in qa_train['macro_domain'].tolist()])
class_counts = np.bincount(train_label_ids, minlength=len(labels))

# standard i.i.d. mini-batches with shuffle=True; no WeightedRandomSampler (no class-frequency-based resampling).
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=_num_workers, pin_memory=True
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_num_workers, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))
print('Class counts:', class_counts.tolist())

23311 2841 2991
Class counts: [1167, 5580, 960, 3903, 2409, 888, 4515, 3889]


In [9]:
# TextCNN (Kim 2014): multi-width 1D conv + ReLU + max-over-time + dropout + linear head.
# default filter_sizes (3,4,5), num_filters=100, dropout=0.5 match the paper / repo README.
# embed_dropout: extra regularization on token vectors (not in Kim 2014; helps reduce train/val gap).
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, filter_sizes=(3,4,5), num_filters=100, dropout=0.5, embed_dropout=0.0, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_dropout = nn.Dropout(embed_dropout) if embed_dropout and embed_dropout > 0 else None
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x)              # (B, L, D)
        if self.embed_dropout is not None:
            emb = self.embed_dropout(emb)
        emb = emb.transpose(1, 2)           # (B, D, L)
        feats = [F.relu(conv(emb)) for conv in self.convs]
        pooled = [F.max_pool1d(f, kernel_size=f.shape[2]).squeeze(2) for f in feats]
        out = torch.cat(pooled, dim=1)
        out = self.dropout(out)
        return self.fc(out)


class FocalLoss(nn.Module):
    """Multi-class focal loss with optional class weights.
    Per sample: w[y] * (1 - p_y)^gamma * CE(sample), CE = cross_entropy(..., reduction='none').
    p_y is softmax probability of the true class (computed in float32 for stability under AMP).
    """

    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma = float(gamma)
        self.label_smoothing = float(label_smoothing)
        if weight is not None:
            self.register_buffer('class_weight', weight)
        else:
            self.class_weight = None

    def forward(self, logits, targets):
        w = self.class_weight
        ce = F.cross_entropy(
            logits, targets, weight=w, reduction='none', label_smoothing=self.label_smoothing
        )
        pt = F.softmax(logits.float(), dim=1).gather(1, targets.unsqueeze(1)).squeeze(1).clamp_min(1e-8)
        mod = (1.0 - pt) ** self.gamma
        return (mod * ce).mean()

In [10]:
# L2 max-norm on final linear layer weights after each optimizer step (Kim 2014).
# Small weight_decay is L2 on all params (not in Kim 2014); use 0.0 to match paper strictly.
MAX_NORM_FC = 3.0  # Kim (2014): L2 max-norm tren trong so lop softmax (fc)
WEIGHT_DECAY = 1e-4
TEXTCNN_DROPOUT = 0.55  # post-pool dropout; 0.5 = Kim default; slightly higher often reduces overfitting.
EMBED_DROPOUT = 0.2
LABEL_SMOOTHING = 0.05  # softens logits; reduces overconfidence vs CE-only (not in Kim 2014).
# Loss head: weighted CE vs focal (same class weights). Focal down-weights easy rows; try gamma 1.5-2.5.
USE_FOCAL = True
FOCAL_GAMMA = 2.0

model = TextCNN(
    vocab_size=len(stoi),
    # learned embedding dim (paper used 300d Word2Vec; 200 is a lighter default here).
    embed_dim=200,
    num_classes=len(labels),
    # explicit Kim (2014) conv widths / filter count / dropout (same as class defaults).
    filter_sizes=(3, 4, 5),
    num_filters=100,
    dropout=TEXTCNN_DROPOUT,
    embed_dropout=EMBED_DROPOUT,
    pad_idx=stoi["<PAD>"],
).to(device)

# Weighted CE: higher weight on rare *true* classes => larger gradient when those rows are misclassified.
# Base = sklearn-balanced: w_c \propto N / (C * n_c). CE_MINORITY_GAMMA > 1 sharpens ratio (stronger minority penalty).
# gamma=1.5 often memorizes minority train patterns; 1.0 is pure balanced and usually generalizes better.
CE_MINORITY_GAMMA = 1.0  # try 1.0 (balanced) or 1.2-1.5 if minority recall is still too low
_counts = torch.as_tensor(class_counts, dtype=torch.float32, device=device).clamp(min=1.0)
_inv_balanced = _counts.sum() / (len(labels) * _counts)
class_weights = _inv_balanced.pow(CE_MINORITY_GAMMA)
class_weights = class_weights * (len(labels) / class_weights.sum())
if USE_FOCAL:
    criterion = FocalLoss(class_weights, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
criterion = criterion.to(device)
print(
    ('FocalLoss' if USE_FOCAL else 'CrossEntropyLoss'),
    f'gamma={FOCAL_GAMMA}' if USE_FOCAL else '',
    '| class_weights:',
    {lab: round(w.item(), 4) for lab, w in zip(labels, class_weights)},
)
# Adadelta as in Kim (2014); do not swap for Adam unless you intentionally change the setup.
optimizer = torch.optim.Adadelta(model.parameters(), lr=1.0, rho=0.95, eps=1e-6, weight_decay=WEIGHT_DECAY)
# mixed precision only (speed/memory); not gradient clipping (see train loop: no clip_grad_norm_).
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

FocalLoss gamma=2.0 | class_weights: {'Civil & Investment': 1.5745, 'Finance & Banking': 0.3293, 'Industry, Resources & Environment': 1.914, 'Justice & Dispute Resolution': 0.4708, 'Labor & Insurance': 0.7627, 'Security & Defense': 2.0692, 'State Organization & Admin': 0.407, 'Transportation': 0.4725}


In [11]:
def run_epoch(model, loader, optimizer=None):
    #one epoch train or eval; uses autocast/GradScaler on CUDA; intentionally no gradient clipping (not in Kim 2014).
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                #project fc weight rows onto L2 ball of radius MAX_NORM_FC (Kim 2014 max-norm constraint).
                with torch.no_grad():
                    w = model.fc.weight
                    norm = w.norm(2)
                    if norm > MAX_NORM_FC:
                        w.mul_(MAX_NORM_FC / (norm + 1e-6))

        total_loss += loss.item() * x.size(0)

        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.detach().cpu().tolist())
        y_pred.extend(pred.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    return avg_loss, macro_f1, y_true, y_pred

In [12]:
EPOCHS = 20
PATIENCE = 5
# Checkpoint + early stopping: higher monitor is better (see _val_monitor).
# - 'composite': val_f1 - VAL_LOSS_WEIGHT * val_loss (catches val loss blowing up while F1 wiggles).
# - 'val_loss': minimize val loss only.
# - 'val_f1': maximize val macro F1 only (legacy).
EARLY_STOP_METRIC = 'composite'
VAL_LOSS_WEIGHT = 0.12  # composite only; try 0.08-0.18 if val_loss ~0.8-2.0
MIN_DELTA = 1e-5


def _val_monitor(va_f1: float, va_loss: float) -> float:
    if EARLY_STOP_METRIC == 'val_f1':
        return float(va_f1)
    if EARLY_STOP_METRIC == 'val_loss':
        return -float(va_loss)
    return float(va_f1) - VAL_LOSS_WEIGHT * float(va_loss)


best_monitor = float('-inf')
best_val_f1_at_ckpt = 0.0
best_val_loss_at_ckpt = float('inf')
wait = 0
best_path = ARTIFACT_DIR / 'textcnn_best.pt'

history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
    va_loss, va_f1, _, _ = run_epoch(model, val_loader, optimizer=None)

    monitor = _val_monitor(va_f1, va_loss)
    history.append({
        'epoch': epoch,
        'train_loss': tr_loss,
        'train_f1': tr_f1,
        'val_loss': va_loss,
        'val_f1': va_f1,
        'val_monitor': monitor,
    })
    print(
        f'Epoch {epoch:02d} | train_loss={tr_loss:.4f} train_f1={tr_f1:.4f} | '
        f'val_loss={va_loss:.4f} val_f1={va_f1:.4f} | monitor={monitor:.4f}'
    )

    if monitor > best_monitor + MIN_DELTA:
        best_monitor = monitor
        best_val_f1_at_ckpt = va_f1
        best_val_loss_at_ckpt = va_loss
        wait = 0
        torch.save(model.state_dict(), best_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print('Early stopping triggered.')
            break

print(
    f'Best checkpoint | metric={EARLY_STOP_METRIC} monitor={round(best_monitor, 4)} | '
    f'val_f1={round(best_val_f1_at_ckpt, 4)} val_loss={round(best_val_loss_at_ckpt, 4)}'
)

Epoch 01 | train_loss=0.4037 train_f1=0.6999 | val_loss=0.8455 val_f1=0.5494 | monitor=0.4479
Epoch 02 | train_loss=0.1761 train_f1=0.8561 | val_loss=0.6836 val_f1=0.5732 | monitor=0.4911
Epoch 03 | train_loss=0.1326 train_f1=0.8925 | val_loss=0.7410 val_f1=0.5597 | monitor=0.4708
Epoch 04 | train_loss=0.0940 train_f1=0.9221 | val_loss=0.8004 val_f1=0.5634 | monitor=0.4674
Epoch 05 | train_loss=0.0788 train_f1=0.9321 | val_loss=0.9157 val_f1=0.5555 | monitor=0.4456
Epoch 06 | train_loss=0.0637 train_f1=0.9440 | val_loss=0.9856 val_f1=0.5565 | monitor=0.4382
Epoch 07 | train_loss=0.0549 train_f1=0.9515 | val_loss=0.7104 val_f1=0.5616 | monitor=0.4763
Early stopping triggered.
Best checkpoint | metric=composite monitor=0.4911 | val_f1=0.5732 val_loss=0.6836


In [13]:
model.load_state_dict(torch.load(best_path, map_location=device))
te_loss, te_f1, y_true, y_pred = run_epoch(model, test_loader, optimizer=None)
print('Test loss:', round(te_loss, 4), '| Test macro F1:', round(te_f1, 4))
print(classification_report(y_true, y_pred, target_names=labels, digits=4, zero_division=0))

Test loss: 0.8685 | Test macro F1: 0.6066
                                   precision    recall  f1-score   support

               Civil & Investment     0.1774    0.0667    0.0969       165
                Finance & Banking     0.9097    0.7341    0.8125       741
Industry, Resources & Environment     0.8724    0.8261    0.8486       207
     Justice & Dispute Resolution     0.7049    0.9698    0.8164       729
                Labor & Insurance     0.5714    0.6667    0.6154        18
               Security & Defense     0.6429    0.0405    0.0763       222
       State Organization & Admin     0.6178    0.7619    0.6824       609
                   Transportation     0.8439    0.9733    0.9040       300

                         accuracy                         0.7389      2991
                        macro avg     0.6676    0.6299    0.6066      2991
                     weighted avg     0.7289    0.7389    0.7033      2991



In [14]:
# metadata mirrors training choices (optimizer, no grad clip, max-norm, sampling, loss) for reproducible inference.
metadata_out = {
    'version': 'textcnn',
    'tokenizer_backend': TOKENIZER_BACKEND,
    'max_len': MAX_LEN,
    'max_vocab_cap': MAX_VOCAB,
    'labels': labels,
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},
    'pad_token': PAD,
    'unk_token': UNK,
    'train_strategy': {
        'sampling': 'shuffle',
        'loss': 'focal_class_weighted' if USE_FOCAL else 'cross_entropy_class_weighted',
        'focal_gamma': FOCAL_GAMMA if USE_FOCAL else None,
        'ce_minority_gamma': CE_MINORITY_GAMMA,
        'ce_class_weights': {lab: round(float(w), 6) for lab, w in zip(labels, class_weights.detach().cpu())},
        'optimizer': 'adadelta',
        'adadelta_lr': 1.0,
        'adadelta_rho': 0.95,
        'adadelta_eps': 1e-6,
        'batch_size': BATCH_SIZE,
        'grad_clip': 'none',
        'max_norm_fc': MAX_NORM_FC,
        'label_smoothing': LABEL_SMOOTHING,
        'weight_decay': WEIGHT_DECAY,
        'dropout': TEXTCNN_DROPOUT,
        'embed_dropout': EMBED_DROPOUT,
        'scheduler': 'none',
        'early_stop_metric': EARLY_STOP_METRIC,
        'early_stop_patience': PATIENCE,
        'early_stop_min_delta': MIN_DELTA,
        'early_stop_val_loss_weight': VAL_LOSS_WEIGHT if EARLY_STOP_METRIC == 'composite' else None,
        'best_val_monitor': round(float(best_monitor), 6),
        'best_val_f1_at_ckpt': round(float(best_val_f1_at_ckpt), 6),
        'best_val_loss_at_ckpt': round(float(best_val_loss_at_ckpt), 6),
    },
}
with open(ARTIFACT_DIR / 'tokenizer_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'stoi': stoi, 'itos': {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False)
with open(ARTIFACT_DIR / 'textcnn_meta.json', 'w', encoding='utf-8') as f:
    json.dump(metadata_out, f, ensure_ascii=False, indent=2)

pd.DataFrame(history).to_csv(ARTIFACT_DIR / 'train_history.csv', index=False)
print('Saved artifacts at:', ARTIFACT_DIR)

Saved artifacts at: /kaggle/working/textcnn_artifacts
